# PROMPT 2: TRAINING & EVALUATION

**Models**: Faster R-CNN, YOLOv8, DETR

**Task**: PCB Defect Detection (6 classes)

**Datasets**: HRIPCB_UPDATE, DeepPCB, DsPCBSD

## 1. Install & Import

In [1]:
# !pip install ultralytics transformers timm
# !pip install torch torchvision --index-url https://download.pytorch.org/whl/cu124

In [1]:
import os, shutil, random, yaml, json, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from tqdm.notebook import tqdm
from collections import defaultdict
from PIL import Image
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from ultralytics import YOLO
from transformers import DetrForObjectDetection, DetrImageProcessor

# Seed & Device
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

NUM_CLASSES = 6
CLASS_NAMES = ['Missing_hole', 'Mouse_bite', 'Open_circuit', 'Short', 'Spurious_copper', 'Spur']
IMG_SIZE = 640
NUM_EPOCHS = 10

Device: cuda
GPU: NVIDIA GeForce RTX 3050 Laptop GPU
VRAM: 4.3 GB


## 2. Paths & Data

In [2]:
ROOT = Path(r'E:\Machine Learning\Deep Learning\project\DEEP\DEEP')
COMBINED = ROOT / 'combined'

DATASETS = {
    'HRIPCB_UPDATE': (ROOT / 'HRIPCB_UPDATE', {'train': 'train', 'val': 'val', 'test': 'test'}),
    'DeepPCB':       (ROOT / 'DeepPCB',       {'train': 'train', 'val': 'valid', 'test': 'test'}),
    'DsPCBSD':       (ROOT / 'PCB.v1i.yolov8', {'train': 'train', 'val': 'valid'}),
}

# Verify
for name, (path, _) in DATASETS.items():
    print(f"{'OK' if path.exists() else 'MISSING'} {name}: {path}")

OK HRIPCB_UPDATE: E:\Machine Learning\Deep Learning\project\DEEP\DEEP\HRIPCB_UPDATE
OK DeepPCB: E:\Machine Learning\Deep Learning\project\DEEP\DEEP\DeepPCB
OK DsPCBSD: E:\Machine Learning\Deep Learning\project\DEEP\DEEP\PCB.v1i.yolov8


## 3. Gop Datasets

In [3]:
# Gop 3 datasets vao thu muc combined
for split in ['train', 'val', 'test']:
    (COMBINED / split / 'images').mkdir(parents=True, exist_ok=True)
    (COMBINED / split / 'labels').mkdir(parents=True, exist_ok=True)

for ds_name, (ds_path, splits) in DATASETS.items():
    for split, folder in splits.items():
        src_imgs = ds_path / folder / 'images'
        src_lbls = ds_path / folder / 'labels'
        if not src_imgs.exists():
            continue
        for f in src_imgs.glob('*'):
            if f.suffix.lower() in ['.jpg', '.png', '.jpeg']:
                dst = COMBINED / split / 'images' / f'{ds_name}_{f.name}'
                if not dst.exists():
                    shutil.copy2(f, dst)
                    lbl = src_lbls / (f.stem + '.txt')
                    if lbl.exists():
                        shutil.copy2(lbl, COMBINED / split / 'labels' / f'{ds_name}_{f.stem}.txt')

for split in ['train', 'val', 'test']:
    n = len(list((COMBINED / split / 'images').glob('*')))
    print(f'{split}: {n} images')

# YOLO data.yaml
yaml_cfg = {'path': str(COMBINED), 'train': 'train/images', 'val': 'val/images',
            'test': 'test/images', 'nc': NUM_CLASSES, 'names': CLASS_NAMES}
with open(COMBINED / 'data.yaml', 'w') as f:
    yaml.dump(yaml_cfg, f)
print('data.yaml saved!')

train: 19698 images
val: 1642 images
test: 0 images
data.yaml saved!


## 4. Dataset

In [4]:
class PCBDataset(Dataset):
    def __init__(self, root, split, img_size=640):
        self.img_size = img_size
        self.imgs_dir = Path(root) / split / 'images'
        self.lbls_dir = Path(root) / split / 'labels'
        self.files = sorted([f for f in self.imgs_dir.glob('*') if f.suffix.lower() in ['.jpg','.png','.jpeg']])
        self.tf = transforms.Compose([transforms.Resize((img_size, img_size)), transforms.ToTensor()])
        print(f'{split}: {len(self.files)} images')

    def __len__(self): return len(self.files)

    def __getitem__(self, idx):
        img = self.tf(Image.open(self.files[idx]).convert('RGB'))
        boxes, labels = [], []
        lbl_path = self.lbls_dir / (self.files[idx].stem + '.txt')
        if lbl_path.exists():
            for line in open(lbl_path):
                p = line.strip().split()
                if len(p) >= 5:
                    c = int(p[0]) % NUM_CLASSES
                    cx, cy, w, h = float(p[1]), float(p[2]), float(p[3]), float(p[4])
                    x1, y1 = max(0,(cx-w/2))*self.img_size, max(0,(cy-h/2))*self.img_size
                    x2, y2 = min(1,(cx+w/2))*self.img_size, min(1,(cy+h/2))*self.img_size
                    if x2 > x1+1 and y2 > y1+1:
                        boxes.append([x1,y1,x2,y2])
                        labels.append(c + 1)
        target = {
            'boxes': torch.tensor(boxes, dtype=torch.float32) if boxes else torch.zeros((0,4)),
            'labels': torch.tensor(labels, dtype=torch.int64) if labels else torch.zeros((0,), dtype=torch.int64),
            'image_id': torch.tensor([idx]),
        }
        return img, target

collate_fn = lambda b: tuple(zip(*b))

train_ds = PCBDataset(COMBINED, 'train')
val_ds   = PCBDataset(COMBINED, 'val')
test_ds  = PCBDataset(COMBINED, 'test')

train_loader = DataLoader(train_ds, batch_size=4, shuffle=True, num_workers=2, collate_fn=collate_fn)
val_loader   = DataLoader(val_ds,   batch_size=4, shuffle=False, num_workers=2, collate_fn=collate_fn)
test_loader  = DataLoader(test_ds,  batch_size=4, shuffle=False, num_workers=2, collate_fn=collate_fn)

train: 19698 images
val: 1642 images
test: 0 images


## 5. Faster R-CNN

ResNet50-FPN backbone | Two-stage detector | 7 classes (6 + background)

In [ ]:
# Load pre-trained Faster R-CNN va thay head cho 7 classes
faster_rcnn = fasterrcnn_resnet50_fpn(weights='DEFAULT')
in_features = faster_rcnn.roi_heads.box_predictor.cls_score.in_features
faster_rcnn.roi_heads.box_predictor = FastRCNNPredictor(in_features, NUM_CLASSES + 1)
faster_rcnn.to(device)

# Train
optimizer = optim.SGD([p for p in faster_rcnn.parameters() if p.requires_grad], lr=0.005, momentum=0.9, weight_decay=0.0005)
frcnn_losses = []

for epoch in range(NUM_EPOCHS):
    faster_rcnn.train()
    total_loss, n = 0, 0
    for images, targets in tqdm(train_loader, desc=f'FRCNN Epoch {epoch+1}'):
        imgs = [img.to(device) for img in images]
        tgts = [{k: v.to(device) for k, v in t.items()} for t in targets]
        # Chi lay samples co boxes
        pairs = [(i, t) for i, t in zip(imgs, tgts) if len(t['boxes']) > 0]
        if not pairs: continue
        loss_dict = faster_rcnn([p[0] for p in pairs], [p[1] for p in pairs])
        loss = sum(loss_dict.values())
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item(); n += 1
    avg = total_loss / max(n, 1)
    frcnn_losses.append(avg)
    print(f'  Loss: {avg:.4f}')

print('Faster R-CNN training done!')

FRCNN Epoch 1:   0%|          | 0/4925 [00:00<?, ?it/s]

## 6. YOLOv8

CSPDarknet backbone | One-stage detector | Ultralytics

In [5]:
# Load YOLOv8 nano va train
yolo_model = YOLO('yolov8n.pt')

yolo_model.train(
    data=str(COMBINED / 'data.yaml'),
    epochs=NUM_EPOCHS,
    imgsz=IMG_SIZE,
    batch=8,
    device=0 if torch.cuda.is_available() else 'cpu',
    project=str(ROOT / 'runs'),
    name='yolov8_pcb',
    exist_ok=True,
    seed=SEED
)
print('YOLOv8 training done!')

New https://pypi.org/project/ultralytics/8.4.15 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.10  Python-3.13.5 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 3050 Laptop GPU, 4096MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=E:\Machine Learning\Deep Learning\project\DEEP\DEEP\combined\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, moment

KeyboardInterrupt: 

## 7. DETR

ResNet50 backbone | Transformer detector | HuggingFace

In [ ]:
# Load DETR pre-trained va fine-tune cho 6 classes
detr_processor = DetrImageProcessor.from_pretrained('facebook/detr-resnet-50')
detr_model = DetrForObjectDetection.from_pretrained(
    'facebook/detr-resnet-50', num_labels=NUM_CLASSES, ignore_mismatched_sizes=True
).to(device)

# DETR Dataset wrapper
class DETRWrapper(Dataset):
    def __init__(self, base_ds, processor):
        self.base, self.proc = base_ds, processor
    def __len__(self): return len(self.base)
    def __getitem__(self, idx):
        img, tgt = self.base[idx]
        pil = transforms.ToPILImage()(img)
        annots = []
        for i in range(len(tgt['boxes'])):
            x1, y1, x2, y2 = tgt['boxes'][i].tolist()
            annots.append({'bbox': [x1, y1, x2-x1, y2-y1], 'category_id': tgt['labels'][i].item()-1,
                           'area': (x2-x1)*(y2-y1), 'iscrowd': 0})
        enc = self.proc(images=pil,
                        annotations={'image_id': idx, 'annotations': annots,
                                     'orig_size': [IMG_SIZE, IMG_SIZE], 'size': [IMG_SIZE, IMG_SIZE]},
                        return_tensors='pt')
        return enc['pixel_values'].squeeze(), enc['labels'][0] if 'labels' in enc else {}

detr_train_loader = DataLoader(DETRWrapper(train_ds, detr_processor), batch_size=4, shuffle=True,
                               collate_fn=lambda b: (torch.stack([x[0] for x in b]), [x[1] for x in b]))

detr_val_loader = DataLoader(DETRWrapper(val_ds, detr_processor), batch_size=4, shuffle=False,
                             collate_fn=lambda b: (torch.stack([x[0] for x in b]), [x[1] for x in b]))

# Train DETR
optimizer = optim.AdamW(detr_model.parameters(), lr=1e-4, weight_decay=1e-4)
detr_losses = []

for epoch in range(NUM_EPOCHS):
    detr_model.train()
    total_loss, n = 0, 0
    for pv, labels in tqdm(detr_train_loader, desc=f'DETR Epoch {epoch+1}'):
        pv = pv.to(device)
        labels = [{k: v.to(device) if isinstance(v, torch.Tensor) else v for k, v in t.items()} for t in labels]
        out = detr_model(pixel_values=pv, labels=labels)
        optimizer.zero_grad()
        out.loss.backward()
        torch.nn.utils.clip_grad_norm_(detr_model.parameters(), 0.1)
        optimizer.step()
        total_loss += out.loss.item(); n += 1
    avg = total_loss / max(n, 1)
    detr_losses.append(avg)
    print(f'  Loss: {avg:.4f}')

print('DETR training done!')

## 8. Evaluation

In [ ]:
def compute_iou(b1, b2):
    x1, y1 = max(b1[0],b2[0]), max(b1[1],b2[1])
    x2, y2 = min(b1[2],b2[2]), min(b1[3],b2[3])
    inter = max(0,x2-x1)*max(0,y2-y1)
    union = (b1[2]-b1[0])*(b1[3]-b1[1]) + (b2[2]-b2[0])*(b2[3]-b2[1]) - inter
    return inter/union if union > 0 else 0

def eval_preds(all_preds, all_targets, iou_thresh=0.5):
    stats = defaultdict(lambda: {'tp':0,'fp':0,'fn':0})
    for preds, gts in zip(all_preds, all_targets):
        pb = preds['boxes'].cpu().numpy() if len(preds['boxes'])>0 else []
        pl = preds['labels'].cpu().numpy() if len(preds['labels'])>0 else []
        ps = preds.get('scores', torch.ones(len(pl))).cpu().numpy()
        gb = gts['boxes'].cpu().numpy() if len(gts['boxes'])>0 else []
        gl = gts['labels'].cpu().numpy() if len(gts['labels'])>0 else []
        matched = set()
        if len(ps)>0:
            order = np.argsort(-ps)
            pb, pl = (pb[order] if len(pb)>0 else pb), pl[order]
        for i in range(len(pl)):
            best_iou, best_j = 0, -1
            for j in range(len(gl)):
                if j in matched: continue
                if len(pb)>0 and len(gb)>0:
                    iou = compute_iou(pb[i], gb[j])
                    if iou > best_iou and pl[i]==gl[j]: best_iou, best_j = iou, j
            if best_iou >= iou_thresh: stats[pl[i]]['tp'] += 1; matched.add(best_j)
            else: stats[pl[i]]['fp'] += 1
        for j in range(len(gl)):
            if j not in matched: stats[gl[j]]['fn'] += 1

    per_class = {}
    for c in range(1, NUM_CLASSES+1):
        tp,fp,fn = stats[c]['tp'], stats[c]['fp'], stats[c]['fn']
        p = tp/(tp+fp) if tp+fp>0 else 0
        r = tp/(tp+fn) if tp+fn>0 else 0
        f = 2*p*r/(p+r) if p+r>0 else 0
        per_class[c] = {'precision':p, 'recall':r, 'f1':f, 'ap':p*r}
    return {
        'per_class': per_class,
        'mAP': np.mean([v['ap'] for v in per_class.values()]),
        'precision': np.mean([v['precision'] for v in per_class.values()]),
        'recall': np.mean([v['recall'] for v in per_class.values()]),
        'f1': np.mean([v['f1'] for v in per_class.values()]),
    }

## 9. Evaluate All Models

In [ ]:
# ---- Faster R-CNN ----
faster_rcnn.eval()
preds, targets, times = [], [], []
with torch.no_grad():
    for imgs, tgts in tqdm(test_loader, desc='Eval FRCNN'):
        imgs_d = [i.to(device) for i in imgs]
        t0 = time.time()
        outs = faster_rcnn(imgs_d)
        times.append((time.time()-t0)*1000/len(imgs))
        for o, t in zip(outs, tgts):
            mask = o['scores'] >= 0.5
            preds.append({k: v[mask] for k, v in o.items()})
            targets.append(t)
frcnn_res = eval_preds(preds, targets)
frcnn_res['inference_time'] = np.mean(times)
print(f'Faster R-CNN  mAP: {frcnn_res["mAP"]*100:.2f}%')

# ---- YOLOv8 ----
best_pt = ROOT / 'runs' / 'yolov8_pcb' / 'weights' / 'best.pt'
if not best_pt.exists(): best_pt = ROOT / 'runs' / 'yolov8_pcb' / 'weights' / 'last.pt'
yolo_val = YOLO(str(best_pt))
metrics = yolo_val.val(data=str(COMBINED / 'data.yaml'), imgsz=IMG_SIZE, verbose=False)
yolo_res = {
    'mAP': float(metrics.box.map50),
    'precision': float(metrics.box.mp),
    'recall': float(metrics.box.mr),
    'f1': 2*float(metrics.box.mp)*float(metrics.box.mr)/(float(metrics.box.mp)+float(metrics.box.mr)+1e-8),
    'inference_time': metrics.speed.get('inference', 0) if hasattr(metrics, 'speed') else 0,
    'per_class': {i+1: {'ap': float(metrics.box.ap50[i]), 'precision': float(metrics.box.p[i]),
                        'recall': float(metrics.box.r[i]),
                        'f1': 2*float(metrics.box.p[i])*float(metrics.box.r[i])/(float(metrics.box.p[i])+float(metrics.box.r[i])+1e-8)}
                  for i in range(min(NUM_CLASSES, len(metrics.box.ap50)))}
}
print(f'YOLOv8        mAP: {yolo_res["mAP"]*100:.2f}%')

# ---- DETR ----
detr_model.eval()
preds, targets, times = [], [], []
with torch.no_grad():
    for pv, labels in tqdm(detr_val_loader, desc='Eval DETR'):
        pv = pv.to(device)
        t0 = time.time()
        out = detr_model(pixel_values=pv)
        times.append((time.time()-t0)*1000/pv.size(0))
        for b in range(pv.size(0)):
            probs = out.logits[b].softmax(-1)
            scores, pred_l = probs[:, :-1].max(-1)
            mask = scores >= 0.5
            bx = out.pred_boxes[b][mask]
            cx,cy,w,h = bx.unbind(-1)
            xyxy = torch.stack([(cx-w/2)*IMG_SIZE,(cy-h/2)*IMG_SIZE,(cx+w/2)*IMG_SIZE,(cy+h/2)*IMG_SIZE], -1)
            preds.append({'boxes': xyxy, 'labels': pred_l[mask]+1, 'scores': scores[mask]})
            tgt = labels[b]
            if 'class_labels' in tgt:
                gb = tgt['boxes']; gcx,gcy,gw,gh = gb.unbind(-1)
                gxyxy = torch.stack([(gcx-gw/2)*IMG_SIZE,(gcy-gh/2)*IMG_SIZE,(gcx+gw/2)*IMG_SIZE,(gcy+gh/2)*IMG_SIZE], -1)
                targets.append({'boxes': gxyxy, 'labels': tgt['class_labels']+1})
            else:
                targets.append({'boxes': torch.zeros(0,4), 'labels': torch.zeros(0, dtype=torch.long)})
detr_res = eval_preds(preds, targets)
detr_res['inference_time'] = np.mean(times)
print(f'DETR          mAP: {detr_res["mAP"]*100:.2f}%')

all_results = {'Faster R-CNN': frcnn_res, 'YOLOv8': yolo_res, 'DETR': detr_res}

## 10. Table 1: Per-Class AP@0.5

In [ ]:
rows = {}
for model, res in all_results.items():
    rows[model] = {CLASS_NAMES[c-1]: f"{res['per_class'].get(c,{}).get('ap',0)*100:.2f}%"
                   for c in range(1, NUM_CLASSES+1)}
df1 = pd.DataFrame(rows).T
df1.index.name = 'Model'
print(df1.to_string())
df1.to_csv(ROOT / 'table1_defect_performance.csv')

## 11. Table 2: Model Comparison

In [ ]:
rows = []
for name, r in all_results.items():
    rows.append({'Model': name, 'mAP@0.5': f"{r['mAP']*100:.2f}%",
                 'Precision': f"{r['precision']*100:.2f}%", 'Recall': f"{r['recall']*100:.2f}%",
                 'F1': f"{r['f1']:.4f}", 'Inference(ms)': f"{r.get('inference_time',0):.1f}"})
df2 = pd.DataFrame(rows).set_index('Model')
print(df2.to_string())

# Best model
best = max(all_results.items(), key=lambda x: x[1]['mAP'])
print(f'\nBest: {best[0]} (mAP={best[1]["mAP"]*100:.2f}%)')
df2.to_csv(ROOT / 'table2_model_comparison.csv')

## 12. Visualization

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
names = list(all_results.keys())
colors = ['#FF6B6B', '#4ECDC4', '#45B7D1']

# mAP
vals = [all_results[m]['mAP']*100 for m in names]
bars = axes[0,0].bar(names, vals, color=colors, edgecolor='black')
axes[0,0].set_title('mAP@0.5', fontweight='bold', fontsize=14)
axes[0,0].set_ylabel('%'); axes[0,0].grid(axis='y', alpha=0.3)
for b, v in zip(bars, vals): axes[0,0].text(b.get_x()+b.get_width()/2, v+0.3, f'{v:.1f}%', ha='center', fontweight='bold')

# Precision & Recall
x = np.arange(len(names))
axes[0,1].bar(x-0.17, [all_results[m]['precision']*100 for m in names], 0.34, label='Precision', color='#FF6B6B')
axes[0,1].bar(x+0.17, [all_results[m]['recall']*100 for m in names], 0.34, label='Recall', color='#4ECDC4')
axes[0,1].set_xticks(x); axes[0,1].set_xticklabels(names)
axes[0,1].set_title('Precision vs Recall', fontweight='bold', fontsize=14)
axes[0,1].legend(); axes[0,1].grid(axis='y', alpha=0.3)

# Inference Time
inf = [all_results[m].get('inference_time',0) for m in names]
bars = axes[1,0].bar(names, inf, color=colors, edgecolor='black')
axes[1,0].set_title('Inference Time (ms)', fontweight='bold', fontsize=14)
axes[1,0].grid(axis='y', alpha=0.3)
for b, v in zip(bars, inf): axes[1,0].text(b.get_x()+b.get_width()/2, v+0.3, f'{v:.1f}', ha='center', fontweight='bold')

# Loss curves
axes[1,1].plot(frcnn_losses, 'o-', label='Faster R-CNN', color='#FF6B6B', lw=2)
axes[1,1].plot(detr_losses, 's-', label='DETR', color='#45B7D1', lw=2)
axes[1,1].set_title('Training Loss', fontweight='bold', fontsize=14)
axes[1,1].set_xlabel('Epoch'); axes[1,1].set_ylabel('Loss')
axes[1,1].legend(); axes[1,1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(ROOT / 'model_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
x = np.arange(NUM_CLASSES)
w = 0.25
for i, (name, res) in enumerate(all_results.items()):
    aps = [res['per_class'].get(c+1,{}).get('ap',0)*100 for c in range(NUM_CLASSES)]
    ax.bar(x + i*w, aps, w, label=name, color=colors[i], edgecolor='black')
ax.set_xticks(x + w); ax.set_xticklabels(CLASS_NAMES, rotation=25, ha='right')
ax.set_title('Per-Class AP@0.5', fontweight='bold', fontsize=14)
ax.set_ylabel('AP (%)'); ax.legend(); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(ROOT / 'per_class_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

## 13. Summary

In [ ]:
print('='*60)
print('DONE! 3 models trained & evaluated'.center(60))
print('='*60)
print(f'\nFaster R-CNN: mAP={frcnn_res["mAP"]*100:.2f}%  F1={frcnn_res["f1"]:.4f}')
print(f'YOLOv8:      mAP={yolo_res["mAP"]*100:.2f}%  F1={yolo_res["f1"]:.4f}')
print(f'DETR:        mAP={detr_res["mAP"]*100:.2f}%  F1={detr_res["f1"]:.4f}')
print(f'\nBest: {max(all_results.items(), key=lambda x: x[1]["mAP"])[0]}')